In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

### 1. 读取数据

In [2]:
base = pd.read_csv('../Dataset/ml-100k/u1.base',sep='\t',names=['user_id','item_id','rate','ts'])
test = pd.read_csv('../Dataset/ml-100k/u1.test',sep='\t',names=['user_id','item_id','rate','ts'])
n_users = base['user_id'].max()
n_items = base['item_id'].max()

In [3]:
R = np.zeros((n_users,n_items),dtype=np.float32)
M = np.zeros((n_users,n_items),dtype=np.float32)
for row in base.itertuples():
    u = row.user_id - 1
    i = row.item_id - 1
    R[u,i] = row.rate
    M[u,i] = 1.0

R_tensor = torch.from_numpy(R)
M_tensor = torch.from_numpy(M)

### 2. 划分交互集与非交互集

In [4]:
interacted = {}
non_interacted = {}
for u in range(n_users):
    interacted[u] = np.where(M[u] == 1)[0].tolist()
    non_interacted[u] = np.where(M[u] == 0)[0].tolist()

"""
M[u]                    [1, 1, 0, 1, 0]        # 第 u 行
M[u] == 1               [T, T, F, T, F]        # 逐个判断是否=1
np.where(...)           (array([0,1,3]),)      # True 的下标，包在元组里
np.where(...)[0]        [0, 1, 3]              # 取出元组里的数组
np.where(...)[0].tolist()  [0, 1, 3]           # 转 Python list
"""

'\nM[u]                    [1, 1, 0, 1, 0]        # 第 u 行\nM[u] == 1               [T, T, F, T, F]        # 逐个判断是否=1\nnp.where(...)           (array([0,1,3]),)      # True 的下标，包在元组里\nnp.where(...)[0]        [0, 1, 3]              # 取出元组里的数组\nnp.where(...)[0].tolist()  [0, 1, 3]           # 转 Python list\n'

### 3. 参数初始化与超参数设置

In [5]:
import random

# T = {100,500,1000}
# alpha_u = alpha_v = beta_v = {0.001,0.01,0.1}
d = 20
mu = M.mean()

b_i = np.zeros((n_items,1))
b_i = M.sum(axis = 0) / n_users - mu
# for i in range(n_items):
#     temp = 0
#     for u in range(n_users):
#         temp += M[u,i]
#     b_i[i] = temp / n_users - mu

V = np.zeros((n_items,d),dtype=np.float32)
U = np.zeros((n_users,d),dtype=np.float32)

for i in range(n_items):
    for k in range(d):
        V[i,k] = (random.random() - 0.5) * 0.01

for u in range(n_users):
    for k in range(d):
        U[u,k] = (random.random() - 0.5) * 0.01


### 4. 二值化

In [6]:
train_pos = base[base['rate'].isin([4, 5])].copy()   # 训练正样本
test_pos  = test[test['rate'].isin([4, 5])].copy()   # 测试正样本
all_items = set(train_pos['item_id']) | set(test_pos['item_id'])

### 5. 训练

In [7]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

pos_pairs = list(zip(*np.where(M == 1))) # 所有正样本 (u,i)
n_pos = len(pos_pairs)

gamma = 0.01        # 学习率
reg = 0.01          # 正则系数
T = 500             # epoch 数

for t in range(T):
    total_loss = 0.0
    for _ in range(n_pos): # 每个 epoch 采 n_pos 个三元组
        # 随机挑正样本 (u,i)，再随机挑负样本 j
        u, i = random.choice(pos_pairs)
        j = random.choice(non_interacted[u])

        # 分数差 x = r_ui - r_uj
        r_ui = np.dot(U[u], V[i]) + b_i[i]
        r_uj = np.dot(U[u], V[j]) + b_i[j]
        x = r_ui - r_uj

        grad_x = -sigmoid(-x)

        U[u]   -= gamma * (grad_x * (V[i] - V[j]) + reg * U[u])
        V[i]   -= gamma * (grad_x * U[u] + reg * V[i])
        V[j]   -= gamma * (- grad_x * U[u] + reg * V[j])
        b_i[i] -= gamma * (grad_x + reg * b_i[i])
        b_i[j] -= gamma * (- grad_x + reg * b_i[j])

        total_loss += -np.log(sigmoid(x) + 1e-15)

    if t % 20 == 0:
        print(f"epoch {t:3d}  平均 loss = {total_loss / n_pos:.4f}")


epoch   0  平均 loss = 0.5700
epoch  20  平均 loss = 0.3562
epoch  40  平均 loss = 0.3290
epoch  60  平均 loss = 0.2531
epoch  80  平均 loss = 0.2244
epoch 100  平均 loss = 0.1983
epoch 120  平均 loss = 0.1829
epoch 140  平均 loss = 0.1719
epoch 160  平均 loss = 0.1610
epoch 180  平均 loss = 0.1529
epoch 200  平均 loss = 0.1461
epoch 220  平均 loss = 0.1413
epoch 240  平均 loss = 0.1376
epoch 260  平均 loss = 0.1342
epoch 280  平均 loss = 0.1318
epoch 300  平均 loss = 0.1313
epoch 320  平均 loss = 0.1259
epoch 340  平均 loss = 0.1270
epoch 360  平均 loss = 0.1278
epoch 380  平均 loss = 0.1265
epoch 400  平均 loss = 0.1242
epoch 420  平均 loss = 0.1256
epoch 440  平均 loss = 0.1242
epoch 460  平均 loss = 0.1238
epoch 480  平均 loss = 0.1231


### 6. 评估 Pre@5 & Rec@5

In [8]:
gt   = test_pos.groupby('user_id')['item_id'].apply(set).to_dict()     # 测试正样本
seen = train_pos.groupby('user_id')['item_id'].apply(set).to_dict()    # 训练已交互

# ---------- BPR 打分 ----------
bpr_pre_list, bpr_rec_list = [], []
for u in gt:
    rel = gt[u]
    seen_u = seen.get(u, set())
    candidates = [i for i in all_items if i not in seen_u]              # 候选 = 物品集 - 已交互
    scored = [(i, float(U[u-1] @ V[i-1] + b_i[i-1])) for i in candidates]
    scored.sort(key=lambda p: -p[1])                                    # 分数降序
    top5 = [i for i, _ in scored[:5]]
    hits = len(set(top5) & rel)
    bpr_pre_list.append(hits / 5)
    bpr_rec_list.append(hits / len(rel) if rel else 0.0)

# ---------- PopRank 打分（物品流行度） ----------
pop = train_pos.groupby('item_id').size().to_dict()
ranked_items = sorted(all_items, key=lambda i: pop.get(i, 0), reverse=True)

pop_pre_list, pop_rec_list = [], []
for u in gt:
    rel = gt[u]
    seen_u = seen.get(u, set())
    rec = [i for i in ranked_items if i not in seen_u]
    top5 = rec[:5]
    hits = len(set(top5) & rel)
    pop_pre_list.append(hits / 5)
    pop_rec_list.append(hits / len(rel) if rel else 0.0)



### 7. 打印结果

In [9]:
print("\n" + "=" * 46)
print("Prediction performance on MovieLens100K (u1.base.OCCF, u1.test.OCCF)")
print("=" * 46)
print(f"{'':<12}{'PopRank':>12}{'BPR':>12}")
print(f"{'Pre@5':<12}{np.mean(pop_pre_list):>12.4f}{np.mean(bpr_pre_list):>12.4f}")
print(f"{'Rec@5':<12}{np.mean(pop_rec_list):>12.4f}{np.mean(bpr_rec_list):>12.4f}")
print("=" * 46)
print("参考值：PopRank 0.2338/0.0571，BPR 0.3864/0.1184")


Prediction performance on MovieLens100K (u1.base.OCCF, u1.test.OCCF)
                 PopRank         BPR
Pre@5             0.2338      0.3044
Rec@5             0.0571      0.1001
参考值：PopRank 0.2338/0.0571，BPR 0.3864/0.1184
